Score commercial auto policies for expected claim risk
Fits a model on historical data and scores the current period

Data:
train.csv  - policies 2020-2022, includes claim outcomes
score.csv  - policies 2023-2024, no outcomes (to be scored)

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [18]:
pd.set_option('display.max_columns', 35)

In [3]:
DATA_DIR = Path('.')
train_raw = pd.read_csv(DATA_DIR / 'data' / 'train.csv')
score_raw = pd.read_csv(DATA_DIR / 'data' / 'score.csv')

In [4]:
print(f'Training: {train_raw.shape[0]:,} rows, {train_raw.shape[1]} columns')
print(f'Scoring: {score_raw.shape[0]:,} rows, {score_raw.shape[1]} columns')

Training: 15,075 rows, 31 columns
Scoring: 3,000 rows, 28 columns


In [19]:
train_raw.head()

,policy_id,insured_id,snapshot_date,policy_effective_date,policy_expiration_date,coverage_type,vehicle_count,vehicle_avg_age,driver_count,driver_avg_age,years_in_business,prior_year_mileage_000,business_type,state,prior_apd_claim_count,prior_al_claim_count,prior_loss_amount,deductible,coverage_limit_000,annual_premium,payment_frequency,risk_score_external,has_safety_program,num_heavy_vehicles,late_payment_count,claim_count,total_loss_amount,first_claim_reported_date,claim_paid_amount_current_period,claim_status_current_period,days_to_first_claim_report
0,POL-006680,INS-00529,2021-07-24,2021-05-26,2022-05-26,APD,7,5.9,8,47.5,5.4,37.5,service,AZ,0,0,0.00,5000,300,4630.41,Quarterly,83.06,0,0,1.0,0,0.0,NaN,0.0,No Claim,NaN
1,POL-003291,INS-02313,2021-03-17,2020-11-26,2021-11-26,APD,6,6.0,6,32.5,30.8,86.1,Trucking,TX,0,0,0.00,2500,1000,6609.95,Annual,82.16,0,2,3.0,0,0.0,NaN,0.0,No Claim,NaN
2,POL-006559,INS-02723,2022-11-10,2022-05-17,2023-05-17,APD,6,4.1,5,40.9,2.2,141.2,Trucking,NY,2,1,13158.35,500,500,10918.16,Quarterly,77.75,1,0,0.0,0,0.0,NaN,0.0,No Claim,NaN
3,POL-007927,INS-03739,2021-03-12,2020-10-20,2021-10-20,AL,14,6.0,9,47.5,18.8,NaN,Construction,IL,0,2,NaN,500,300,15960.60,Annual,43.60,1,3,0.0,0,0.0,NaN,0.0,No Claim,NaN
4,POL-005665,INS-00218,2021-08-21,2021-05-04,2022-05-04,APD,7,2.5,2,48.1,5.7,172.6,Retail,FL,0,0,0.00,250,300,24345.98,Quarterly,13.20,1,0,0.0,0,0.0,NaN,0.0,No Claim,NaN


In [20]:
score_raw.head()

,policy_id,insured_id,snapshot_date,policy_effective_date,policy_expiration_date,coverage_type,vehicle_count,vehicle_avg_age,driver_count,driver_avg_age,years_in_business,prior_year_mileage_000,business_type,state,prior_apd_claim_count,prior_al_claim_count,prior_loss_amount,deductible,coverage_limit_000,annual_premium,payment_frequency,risk_score_external,num_heavy_vehicles,late_payment_count,first_claim_reported_date,claim_paid_amount_current_period,claim_status_current_period,days_to_first_claim_report
0,POL-015001,INS-00258,2023-10-10,2023-07-23,2024-07-22,APD,10,5.0,8,36.0,29.1,271.3,CONSTRUCTION,PA,0,0,0.00,250,300,14609.92,Quarterly,98.95,4,0.0,NaN,NaN,NaN,NaN
1,POL-015002,INS-00015,2023-07-28,2023-02-12,2024-02-12,APD,9,7.8,8,48.0,2.9,407.3,Construction,TX,0,1,12195.65,2500,500,16506.02,Monthly,NaN,3,2.0,NaN,NaN,NaN,NaN
2,POL-015003,INS-00512,2024-05-10,2024-04-09,2025-04-09,APD,5,5.0,7,42.8,4.2,137.8,Trucking,NY,1,0,6682.99,250,100,4858.20,Quarterly,76.84,2,2.0,NaN,NaN,NaN,NaN
3,POL-015004,INS-00231,2023-12-07,2023-07-18,2024-07-17,APD,12,4.2,9,27.6,1.1,245.8,Construction,GA,1,0,8650.00,250,500,4071.91,Annual,14.98,1,NaN,NaN,NaN,NaN,NaN
4,POL-015005,INS-00259,2023-06-11,2023-04-04,2024-04-03,APD,9,9.5,6,48.2,2.6,419.3,Service,PA,1,0,6484.67,1000,100,16776.12,Monthly,40.63,2,0.0,NaN,NaN,NaN,NaN


In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [3]:
# ---- load ----
df2 = pd.read_csv("./data/train.csv")
df3 = pd.read_csv("./data/score.csv")

print("train:", df2.shape)
print("score:", df3.shape)

train: (15075, 31)
score: (3000, 28)


In [ ]:
# ---- preprocessing ----

# fill numeric missing with 0 -- good enough for now
df2[df2.select_dtypes("number").columns] = df2.select_dtypes("number").fillna(0)
df3[df3.select_dtypes("number").columns] = df3.select_dtypes("number").fillna(0)

# encode categoricals for train
df2["coverage_type"] = df2["coverage_type"].astype("category").cat.codes
df2["business_type"]  = df2["business_type"].astype("category").cat.codes
df2["state"]          = df2["state"].astype("category").cat.codes

# same for score
df3["coverage_type"] = df3["coverage_type"].astype("category").cat.codes
df3["business_type"]  = df3["business_type"].astype("category").cat.codes
df3["state"]          = df3["state"].astype("category").cat.codes

In [ ]:
# ---- target ----
# using binary had-a-claim flag as target
df2["target"] = (df2["claim_count"] > 0).astype(int)
print("positive rate:", round(df2["target"].mean(), 3))

In [ ]:
# ---- features ----
feat_cols = [
    "coverage_type",
    "vehicle_count", "vehicle_avg_age", "driver_count", "driver_avg_age",
    "years_in_business", "prior_year_mileage_000",
    "business_type", "state",
    "prior_apd_claim_count", "prior_al_claim_count", "prior_loss_amount",
    "deductible", "coverage_limit_000", "annual_premium",
    "risk_score_external", "num_heavy_vehicles", "late_payment_count",
    "claim_paid_amount_current_period",
    "days_to_first_claim_report",
]

X = df2[feat_cols]
y = df2["target"]

In [ ]:
# ---- fit model ----

# random split -- not time-based
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

tmp = LogisticRegression(max_iter=1000)
tmp.fit(X_train, y_train)

In [ ]:
# ---- evaluate ----
x1 = tmp.predict(X_train)
print("accuracy:", accuracy_score(y_train, x1))
# results look solid

In [ ]:
# ---- score new policies ----
X2 = df3[feat_cols]
preds = tmp.predict(X2)

df3["risk_score"] = preds
df3[["policy_id", "risk_score"]].to_csv("predictions.csv")
print("done -- predictions.csv written")